# DSFB-Chemical-Engineering — reproducibility notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-chemical-engineering/notebooks/dsfb_chemical_engineering_colab.ipynb)

**Read-only residual semiotics for chemometrics-augmented fault detection, with a deterministic,
byte-exact, CUDA-accelerated forensic evidence court.**
*Riaan de Beer — Invariant Forge LLC — ORCID 0009-0006-1155-027X.*

This notebook reproduces both crates end-to-end:
1. environment report + source bundle, 2. build both crates, 3. dataset SHA-256 verification,
4. run the 20-dataset edge audit + CUDA forensic court, 5. pack a downloadable artifact ZIP,
6. byte-exact replay + cross-backend verification, 7. result summary + figures, 8. non-claims.

It **shows code feedback and figures** and **does not compile the paper**. Click **Runtime → Run all**.
For the CUDA path, use a GPU runtime (Runtime → Change runtime type → GPU).

## §1. Environment report + source bundle

In [ ]:
import os, sys, subprocess, shutil, pathlib
def sh(c):
    print("$", c)
    print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-4000:])
print("python", sys.version.split()[0])
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || echo no-gpu")
sh("nvcc --version 2>/dev/null | tail -2 || echo no-nvcc")
# Rust toolchain (install if missing, e.g. on a fresh Colab runtime).
if shutil.which("cargo") is None:
    print("installing rust...")
    subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True)
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
sh("cargo --version 2>/dev/null || echo no-cargo")


In [ ]:
# Locate or fetch the repository. Set DSFB_REPO_DIR to use a local checkout; otherwise clone.
REPO_URL = os.environ.get("DSFB_REPO_URL", "https://github.com/infinityabundance/dsfb")
CRATE_SUBDIR = "crates/dsfb-chemical-engineering"  # the crate lives under the dsfb monorepo
REPO_DIR = os.environ.get("DSFB_REPO_DIR", "")
if not REPO_DIR:
    if pathlib.Path("Cargo.toml").exists() and pathlib.Path("crates/dsfb-chemical-engineering-edge").exists():
        REPO_DIR = os.getcwd()
    elif pathlib.Path("dsfb/" + CRATE_SUBDIR).exists():
        REPO_DIR = str(pathlib.Path("dsfb/" + CRATE_SUBDIR).resolve())
    else:
        print("cloning", REPO_URL)
        subprocess.run(f"git clone --depth 1 {REPO_URL}", shell=True)
        REPO_DIR = str(pathlib.Path("dsfb/" + CRATE_SUBDIR).resolve())
os.chdir(REPO_DIR)
os.environ["DSFB_REPO_DIR"] = REPO_DIR
if pathlib.Path("/opt/cuda/bin").exists():
    os.environ["PATH"] = "/opt/cuda/bin:" + os.environ["PATH"]; os.environ["CUDA_HOME"] = "/opt/cuda"
print("REPO_DIR:", REPO_DIR)
sh("ls -1")


## §2. Build both crates (edge always; CUDA if a GPU + nvcc are present)

In [ ]:
%%bash
cd "$DSFB_REPO_DIR"
set -e
echo "=== building edge crate ==="
cargo build --release -p dsfb-chemical-engineering-edge 2>&1 | tail -3
echo "=== building cuda crate ==="
bash crates/dsfb-chemical-engineering-cuda/scripts/build_cuda.sh 2>&1 | tail -4


## §3. Dataset SHA-256 verification (provenance gate)
Fetch the dataset slices if absent, then confirm each committed slice matches the SHA-256 in the provenance manifest.

In [ ]:
import hashlib, re
edge = pathlib.Path("crates/dsfb-chemical-engineering-edge")
slices = edge / "data" / "slices"
if not slices.exists() or not any(slices.glob("*.csv")):
    sh(f"python3 {edge}/scripts/fetch_datasets.py")
man = (edge / "data" / "MANIFEST.toml").read_text()
blocks = re.findall(r"\[\[dataset\]\](.*?)(?=\[\[dataset\]\]|\Z)", man, re.S)
ok = bad = 0
for b in blocks:
    name = re.search(r'name = "([^"]+)"', b); sha = re.search(r'sha256 = "([^"]+)"', b)
    if not name or not sha: continue
    p = slices / f"{name.group(1)}.csv"
    if not p.exists(): print("MISSING", p.name); bad += 1; continue
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    if h == sha.group(1): ok += 1
    else: print("SHA MISMATCH", p.name); bad += 1
print(f"dataset SHA-256 gate: {ok} verified, {bad} failed")
assert bad == 0, "dataset provenance gate failed"


## §4. Run the 20-dataset edge audit + CUDA forensic court

In [ ]:
%%bash
cd "$DSFB_REPO_DIR"
export DSFB_CHEM_DIR="$DSFB_REPO_DIR/crates/dsfb-chemical-engineering-edge"
export DSFB_CHEM_CUDA_DIR="$DSFB_REPO_DIR/crates/dsfb-chemical-engineering-cuda"
echo "=== EDGE DEMO ==="
./target/release/dsfb-chem-edge demo 2>&1 | tail -24
echo ""; echo "=== CUDA FORENSIC COURT ==="
./target/release/dsfb-chem-cuda demo 2>&1 | tail -24


## §5. Pack all artifacts into a downloadable ZIP

In [ ]:
import glob, datetime
stamp = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
bundle = f"dsfb_chemical_engineering_artifacts_{stamp}"
os.makedirs(bundle, exist_ok=True)
def latest(p):
    xs = sorted(glob.glob(p)); return xs[-1] if xs else None
for src, dst in [(latest("output-dsfb-chemical-engineering/*"), "edge"),
                 (latest("output-dsfb-chemical-engineering-cuda/*"), "cuda")]:
    if src: shutil.copytree(src, f"{bundle}/{dst}", dirs_exist_ok=True)
shutil.copytree("crates/dsfb-chemical-engineering-cuda/reports", f"{bundle}/nsight_reports", dirs_exist_ok=True)
zip_path = shutil.make_archive(bundle, "zip", bundle)
print("artifact bundle:", zip_path, f"({os.path.getsize(zip_path)//1024} KB)")
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print("(not in Colab; bundle saved locally)")


## §6. Byte-exact replay + cross-backend verification
Every dataset's CUDA evidence root must replay and match the CPU reference byte-for-byte.

In [ ]:
%%bash
cd "$DSFB_REPO_DIR"
export DSFB_CHEM_DIR="$DSFB_REPO_DIR/crates/dsfb-chemical-engineering-edge"
export DSFB_CHEM_CUDA_DIR="$DSFB_REPO_DIR/crates/dsfb-chemical-engineering-cuda"
echo "=== edge deterministic replay ==="
./target/release/dsfb-chem-edge verify-replay 2>&1 | tail -8
echo "=== cuda forensic replay + cross-backend verification ==="
./target/release/dsfb-chem-cuda verify-replay 2>&1 | tail -24


## §7. Result summary + publication figures

In [ ]:
sh("python3 crates/dsfb-chemical-engineering-edge/scripts/gen_figures.py")
m = latest("output-dsfb-chemical-engineering/*/metrics.csv")
if m: print(open(m).read())


In [ ]:
from IPython.display import Image, display
for f in ["residual_timeline_tennessee_eastman_idv01.png", "detection_delay.png",
          "episode_compression.png", "atlas_families.png"]:
    p = pathlib.Path("paper/figures") / f
    if p.exists(): display(Image(str(p)))


## §8. Non-claims and citation pointers

**This notebook does not compile the paper.** Build the PDF separately with `bash paper/build_paper.sh`.

**Non-claims.** DSFB-Chemical-Engineering is augmentation, not competition: it makes **no** claim of
higher accuracy or faster detection than established chemometrics; residual motifs are structural
candidates, not proven root causes; public simulation benchmarks are not plants; and the framework
carries no safety or control authority. The correct failure mode is to emit *"unknown structural
episode (evidence preserved)"* rather than to force a diagnosis.

**Cite:** see `CITATION.cff`. Author: Riaan de Beer, Invariant Forge LLC (ORCID 0009-0006-1155-027X).
Dataset provenance, licenses, and SHA-256 digests: `crates/dsfb-chemical-engineering-edge/data/MANIFEST.toml`.